In [1]:
import torch

if torch.cuda.is_available():
    print("✅ GPU is available")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU not available")


✅ GPU is available
GPU name: NVIDIA GeForce RTX 4050 Laptop GPU


In [9]:
!pip install --upgrade lightgbm

In [4]:
# ==============================
# 📌 STEP 1: Imports
# ==============================
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split

# ==============================
# 📂 STEP 2: Load datasets & embeddings
# ==============================
train_csv = pd.read_csv("dataset/train.csv")
test_csv  = pd.read_csv("dataset/test.csv")

train_text_embs = np.load("train_text_embeddings_fp16_v2.npy")
test_text_embs  = np.load("test_text_embeddings_fp16_v2.npy")

train_img_df = pd.read_csv("train_features.csv")
test_img_df  = pd.read_csv("test_features.csv")

# Ensure sample_id is integer
train_csv['sample_id'] = train_csv['sample_id'].astype(np.int64)
test_csv['sample_id']  = test_csv['sample_id'].astype(np.int64)
train_img_df['sample_id'] = train_img_df['sample_id'].astype(np.int64)
test_img_df['sample_id']  = test_img_df['sample_id'].astype(np.int64)

# ==============================
# 🧩 STEP 3: Align embeddings with CSV
# ==============================
train_merged = train_csv.merge(train_img_df, on="sample_id", how="left")
test_merged  = test_csv.merge(test_img_df,  on="sample_id", how="left")

# Drop non-feature columns
train_img_features = train_merged.drop(columns=['sample_id','price','catalog_content']).values.astype(np.float32)
test_img_features  = test_merged.drop(columns=['sample_id','catalog_content']).values.astype(np.float32)

# Combine text + image embeddings
train_features = np.hstack([train_text_embs.astype(np.float32), train_img_features])
test_features  = np.hstack([test_text_embs.astype(np.float32), test_img_features])

# Target
y = train_merged['price'].values.astype(np.float32)

print("Train features shape:", train_features.shape)
print("Test  features shape:", test_features.shape)

# ==============================
# 🧩 STEP 4: Train LightGBM in chunks to save RAM
# ==============================
chunk_size = 15000
models = []

for i in range(0, len(train_features), chunk_size):
    print(f"Training chunk {i//chunk_size + 1}")
    X_chunk = train_features[i:i+chunk_size]
    y_chunk = y[i:i+chunk_size]
    
    # Split chunk into train/val
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_chunk, y_chunk, test_size=0.1, random_state=42
    )
    
    model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=64,
        random_state=42
    )
    
    model.fit(
        X_train_split, y_train_split,
        eval_set=[(X_val_split, y_val_split)],
        eval_metric="mae"
    )
    
    models.append(model)

# ==============================
# 🧩 STEP 5: Predict on test set
# ==============================
preds = np.mean([m.predict(test_features) for m in models], axis=0)

# Ensure positive prices
preds = np.maximum(preds, 0.01)

# ==============================
# 🧩 STEP 6: Create submission
# ==============================
submission = pd.DataFrame({
    "sample_id": test_csv['sample_id'],
    "price": preds
})
submission.to_csv("submission.csv", index=False)
print("Submission saved: submission.csv")


ValueError: could not convert string to float: 'https://m.media-amazon.com/images/I/51mo8htwTHL.jpg'

In [8]:
import pandas as pd

train_img_df = pd.read_csv("train_features.csv")
test_img_df  = pd.read_csv("test_features.csv")
print(train_img_df.head())
print(train_img_df.dtypes)


          0         1         2         3         4    5         6         7  \
0  0.000000  0.000000  0.390201  0.000000  0.000000  0.0  0.000000  0.000000   
1  0.000000  0.000000  0.326743  0.691742  0.048034  0.0  0.000000  0.000000   
2  0.000000  0.014721  0.172653  0.000000  0.000000  0.0  0.027951  0.000000   
3  0.000000  0.007763  0.007279  0.000000  0.059895  0.0  0.000000  0.000000   
4  0.045635  0.000000  0.000000  0.000000  0.065872  0.0  0.000000  0.006455   

     8         9  ...      2039      2040  2041      2042      2043      2044  \
0  0.0  0.004456  ...  0.000000  0.129256   0.0  0.005268  0.003708  0.000000   
1  0.0  0.000000  ...  0.202578  0.102678   0.0  0.025811  0.000000  0.000000   
2  0.0  0.000000  ...  0.101217  0.235260   0.0  0.000000  0.045090  0.000000   
3  0.0  0.000000  ...  0.128813  0.532518   0.0  0.000000  0.129173  0.000000   
4  0.0  0.001168  ...  0.000000  0.015849   0.0  0.000000  0.052698  0.056507   

   2045  2046  2047  sample_id  

In [12]:
# ========================================
# STEP 1: Imports
# ========================================
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split

# ========================================
# STEP 2: Load CSVs and embeddings
# ========================================
train_csv = pd.read_csv("dataset/train.csv")  # contains sample_id, price, catalog_content
test_csv  = pd.read_csv("dataset/test.csv")

train_img_df = pd.read_csv("train_features.csv")  # image embeddings + sample_id
test_img_df  = pd.read_csv("test_features.csv")

train_text_embs = np.load("train_text_embeddings_fp16_v2.npy").astype(np.float32)
test_text_embs  = np.load("test_text_embeddings_fp16_v2.npy").astype(np.float32)

# ========================================
# STEP 3: Fix sample_id types
# ========================================
train_csv['sample_id'] = train_csv['sample_id'].astype(np.int64)
test_csv['sample_id']  = test_csv['sample_id'].astype(np.int64)
train_img_df['sample_id'] = train_img_df['sample_id'].astype(np.int64)
test_img_df['sample_id']  = test_img_df['sample_id'].astype(np.int64)

# ========================================
# STEP 4: Add missing rows with zero vectors
# ========================================
missing_train_ids = set(train_csv['sample_id']) - set(train_img_df['sample_id'])
for mid in missing_train_ids:
    zero_row = pd.DataFrame([[0]* (train_img_df.shape[1]-1) + [mid]], columns=train_img_df.columns)
    train_img_df = pd.concat([train_img_df, zero_row], ignore_index=True)

missing_test_ids = set(test_csv['sample_id']) - set(test_img_df['sample_id'])
for mid in missing_test_ids:
    zero_row = pd.DataFrame([[0]* (test_img_df.shape[1]-1) + [mid]], columns=test_img_df.columns)
    test_img_df = pd.concat([test_img_df, zero_row], ignore_index=True)

# ========================================
# STEP 5: Align embeddings by sample_id
# ========================================
train_img_df = train_img_df.set_index('sample_id').loc[train_csv['sample_id']].reset_index()
test_img_df  = test_img_df.set_index('sample_id').loc[test_csv['sample_id']].reset_index()

train_img_features = train_img_df.drop(columns=['sample_id']).values.astype(np.float32)
test_img_features  = test_img_df.drop(columns=['sample_id']).values.astype(np.float32)

# Combine text + image embeddings
train_features = np.hstack([train_text_embs, train_img_features])
test_features  = np.hstack([test_text_embs, test_img_features])

y = train_csv['price'].values

print("Train features shape:", train_features.shape)
print("Test  features shape:", test_features.shape)

# ========================================
# STEP 6: Train LightGBM in chunks (RAM-friendly)
# ========================================
chunk_size = 15000
models = []

for i in range(0, len(train_features), chunk_size):
    print(f"🧩 Training chunk {i//chunk_size + 1}")
    X_chunk = train_features[i:i+chunk_size]
    y_chunk = y[i:i+chunk_size]

    # Split chunk into train/validation
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_chunk, y_chunk, test_size=0.1, random_state=42
    )

    model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=64,
        random_state=42
    )

    # Fit model (older LightGBM versions may not support early_stopping_rounds or verbose)
    model.fit(
        X_train_split, y_train_split,
        eval_set=[(X_val_split, y_val_split)],
        eval_metric="mae"
    )

    models.append(model)

# ========================================
# STEP 7: Predict on test set
# ========================================
preds = np.mean([m.predict(test_features) for m in models], axis=0)

# ========================================
# STEP 8: Create submission
# ========================================
submission = pd.DataFrame({
    "sample_id": test_csv['sample_id'],
    "price": preds
})

submission.to_csv("submission.csv", index=False)
print("✅ Submission file saved!")


Train features shape: (75000, 2816)
Test  features shape: (75000, 2816)
🧩 Training chunk 1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.773955 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717435
[LightGBM] [Info] Number of data points in the train set: 13500, number of used features: 2816
[LightGBM] [Info] Start training from score 23.749263
🧩 Training chunk 2
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.988176 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717431
[LightGBM] [Info] Number of data points in the train set: 13500, number of used features: 2816
[LightGBM] [Info] Start training from score 23.298189
🧩 Training chunk 3
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.906265 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Inf

C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  

✅ Submission file saved!


In [13]:
# SMAPE function
def smape(y_true, y_pred):
    return 100/len(y_true) * np.sum(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

# Example: evaluate on last validation split
y_val_pred = models[-1].predict(X_val_split)
score = smape(y_val_split, y_val_pred)
print(f"SMAPE on last validation chunk: {score:.4f}%")


SMAPE on last validation chunk: 67.7707%


C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [1]:
# ===============================
# 📌 STEP 0: Imports
# ===============================
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
import lightgbm as lgb

# ===============================
# 📌 STEP 1: Load embeddings
# ===============================
train_text_embs = np.load("train_text_embeddings_fp16_v2.npy").astype(np.float32)
test_text_embs  = np.load("test_text_embeddings_fp16_v2.npy").astype(np.float32)

train_img_df = pd.read_csv("train_features.csv")
test_img_df  = pd.read_csv("test_features.csv")

# Ensure image embeddings are float32
img_cols = [c for c in train_img_df.columns if c != 'sample_id']
train_img_features = train_img_df[img_cols].astype(np.float32).values
test_img_features  = test_img_df[img_cols].astype(np.float32).values

# ===============================
# 📌 STEP 2: Align train/test sizes
# ===============================
# Fix any mismatch if needed
min_train = min(train_text_embs.shape[0], train_img_features.shape[0])
train_text_embs = train_text_embs[:min_train]
train_img_features = train_img_features[:min_train]

min_test = min(test_text_embs.shape[0], test_img_features.shape[0])
test_text_embs = test_text_embs[:min_test]
test_img_features = test_img_features[:min_test]

# ===============================
# 📌 STEP 3: Combine embeddings
# ===============================
train_features = np.hstack([train_text_embs, train_img_features])
test_features  = np.hstack([test_text_embs, test_img_features])

# Target (example: price column)
train_csv = pd.read_csv("dataset/train.csv")
y = train_csv['price'].values[:min_train]

# ===============================
# 📌 STEP 4: K-Fold + Chunked Training
# ===============================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
chunk_size = 15000
oof_preds = np.zeros(len(train_features))
test_preds = np.zeros(len(test_features))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train_features)):
    print(f"\n🌟 Fold {fold+1}")
    
    X_tr, X_val = train_features[tr_idx], train_features[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    
    # Train in chunks
    models = []
    for i in range(0, len(X_tr), chunk_size):
        print(f"  Training chunk {i//chunk_size + 1}")
        X_chunk = X_tr[i:i+chunk_size]
        y_chunk = y_tr[i:i+chunk_size]
        
        model = lgb.LGBMRegressor(
            objective="regression",
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=64,
            random_state=42
        )
        model.fit(X_chunk, y_chunk)
        models.append(model)
    
    # OOF prediction for this fold
    fold_oof = np.mean([m.predict(X_val) for m in models], axis=0)
    oof_preds[val_idx] = fold_oof
    
    # Test prediction for this fold
    fold_test = np.mean([m.predict(test_features) for m in models], axis=0)
    test_preds += fold_test / kf.n_splits

# ===============================
# 📌 STEP 5: Evaluate OOF SMAPE
# ===============================
def smape(y_true, y_pred):
    return 100/len(y_true) * np.sum(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

print("\n✅ OOF SMAPE:", smape(y, oof_preds))

# ===============================
# 📌 STEP 6: Save submission
# ===============================
submission = pd.DataFrame({
    'sample_id': test_img_df['sample_id'][:min_test],
    'price': test_preds
})
submission.to_csv("submission.csv", index=False)
print("✅ Submission saved!")



🌟 Fold 1
  Training chunk 1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.918275 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717545
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 2816
[LightGBM] [Info] Start training from score 23.307043
  Training chunk 2
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.868977 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717557
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 2816
[LightGBM] [Info] Start training from score 23.502130
  Training chunk 3
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.874496 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717535
[LightGBM] [Info] Number of data points i

C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  


🌟 Fold 2
  Training chunk 1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.801400 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717564
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 2816
[LightGBM] [Info] Start training from score 23.510272
  Training chunk 2
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.855970 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717555
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 2816
[LightGBM] [Info] Start training from score 23.658972
  Training chunk 3
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.893235 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717567
[LightGBM] [Info] Number of data points i

C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  


🌟 Fold 3
  Training chunk 1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.879815 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717530
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 2816
[LightGBM] [Info] Start training from score 23.464147
  Training chunk 2
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.990719 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717568
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 2816
[LightGBM] [Info] Start training from score 23.782292
  Training chunk 3
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.872617 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717546
[LightGBM] [Info] Number of data points i

...


🌟 Fold 4
  Training chunk 1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.876659 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717544
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 2816
[LightGBM] [Info] Start training from score 23.615819
  Training chunk 2
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.892376 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717573
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 2816
[LightGBM] [Info] Start training from score 23.663739
  Training chunk 3
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.936883 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717548
[LightGBM] [Info] Number of data points i

C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  


🌟 Fold 5
  Training chunk 1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.921348 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717536
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 2816
[LightGBM] [Info] Start training from score 23.334677
  Training chunk 2
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.892283 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717561
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 2816
[LightGBM] [Info] Start training from score 23.645322
  Training chunk 3
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.874524 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 717556
[LightGBM] [Info] Number of data points i

C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  


✅ OOF SMAPE: 69.73937666837232
✅ Submission saved!


In [2]:

# =========================================
# Chunked K-Fold LightGBM pipeline (SMAPE)
# Memory-safe for ~16GB RAM
# =========================================
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, train_test_split
import lightgbm as lgb

# -------------------------
# User-editable file paths
# -------------------------
TRAIN_CSV = "dataset/train.csv"                      # must contain sample_id, price, catalog_content
TEST_CSV  = "dataset/test.csv"
TRAIN_TEXT_EMB = "train_text_embeddings_fp16_v2.npy"   # (75000, 768)
TEST_TEXT_EMB  = "test_text_embeddings_fp16_v2.npy"
TRAIN_IMG_CSV  = "train_features.csv"        # numeric columns 0..2047 + sample_id
TEST_IMG_CSV   = "test_features.csv"

# -------------------------
# Hyperparams
# -------------------------
N_SPLITS = 5
CHUNK_SIZE = 15000        # adjust down if you still see memory problems
LGB_PARAMS = {
    "objective": "regression",
    "n_estimators": 600,
    "learning_rate": 0.05,
    "num_leaves": 64,
    "random_state": 42,
    "verbosity": -1
}

# -------------------------
# Utility: SMAPE (percentage)
# -------------------------
def smape(y_true, y_pred):
    # both arrays on original scale (not log)
    denom = (np.abs(y_true) + np.abs(y_pred))
    # avoid division by zero
    mask = denom == 0
    denom[mask] = 1e-6
    return 100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / denom)

# -------------------------
# 1) Load metadata & embeddings
# -------------------------
print("Loading files...")
train_meta = pd.read_csv(TRAIN_CSV)
test_meta  = pd.read_csv(TEST_CSV)

train_text_embs = np.load(TRAIN_TEXT_EMB).astype(np.float32)
test_text_embs  = np.load(TEST_TEXT_EMB).astype(np.float32)

train_img_df = pd.read_csv(TRAIN_IMG_CSV)   # numeric columns + sample_id
test_img_df  = pd.read_csv(TEST_IMG_CSV)

# -------------------------
# 2) Safety & alignment checks
# -------------------------
# ensure sample_id exists and numeric
for df,name in [(train_meta,'train_meta'), (test_meta,'test_meta'),
                (train_img_df,'train_img_df'), (test_img_df,'test_img_df')]:
    if 'sample_id' not in df.columns:
        raise ValueError(f"Missing sample_id in {name}")
    df['sample_id'] = pd.to_numeric(df['sample_id'], errors='coerce')
    df.dropna(subset=['sample_id'], inplace=True)
    df['sample_id'] = df['sample_id'].astype(np.int64)

# keep order same as train_meta/test_meta
# Merge image features into meta frames (left join), fill missing with 0
train_merged = train_meta.merge(train_img_df, on='sample_id', how='left').fillna(0)
test_merged  = test_meta.merge(test_img_df, on='sample_id', how='left').fillna(0)

# numeric image columns (all columns from image CSV except sample_id)
img_cols = [c for c in train_img_df.columns if c != 'sample_id']

# convert to numpy (but we won't hstack whole arrays at once)
train_img_features = train_merged[img_cols].values.astype(np.float32)
test_img_features  = test_merged[img_cols].values.astype(np.float32)

# quick shape checks
print("shapes:")
print(" train_text_embs:", train_text_embs.shape)
print(" train_img_features:", train_img_features.shape)
print(" test_text_embs:", test_text_embs.shape)
print(" test_img_features:", test_img_features.shape)

# If counts mismatch, align by train_meta order
n_train = len(train_meta)
if train_text_embs.shape[0] != n_train or train_img_features.shape[0] != n_train:
    # Reindex by train_meta.sample_id if needed
    print("Aligning by sample_id order to ensure exact row order...")
    # Build index mapping for text and image if they have sample_id; otherwise assume already aligned
    # Here we assume train_text_embs correspond to train_meta order; if not, user must supply matching arrays
    # For safety, if img rows != n_train, attempt to reorder:
    if train_img_features.shape[0] == n_train:
        pass
    else:
        # If image features length differs, try to reindex from train_img_df using sample_id in train_meta
        # Only do if original train_img_df had sample_id index; otherwise user must correct their files
        try:
            train_img_df_indexed = train_img_df.set_index('sample_id')
            train_img_features = train_meta['sample_id'].map(lambda sid: train_img_df_indexed.loc[sid].values if sid in train_img_df_indexed.index else np.zeros(len(img_cols)))
            # The above mapping returns objects; reconstruct proper ndarray:
            train_img_features = np.vstack([np.array(x, dtype=np.float32).reshape(1,-1) if np.ndim(x)!=1 else x for x in train_img_features])
        except Exception:
            raise RuntimeError("Could not align image features to train_meta; ensure image CSV has one row per sample_id or run pre-processing in Colab to generate aligned embeddings.")

# -------------------------
# 3) Extra lightweight text features
# -------------------------
def extract_text_features(series):
    # assumes series is pandas series of strings
    text_len = series.fillna("").str.len().astype(np.float32).values
    word_count = series.fillna("").str.split().map(len).astype(np.float32).values
    digit_count = series.fillna("").str.count(r'\d').astype(np.float32).values
    avg_word_len = (text_len / (word_count + 1e-6)).astype(np.float32)
    return np.vstack([text_len, word_count, digit_count, avg_word_len]).T  # shape (n,4)

print("Computing small text features...")
train_text_feats = extract_text_features(train_meta['catalog_content'])
test_text_feats  = extract_text_features(test_meta['catalog_content'])

# -------------------------
# 4) Prepare target: use log1p
# -------------------------
y_raw = train_meta['price'].values.astype(np.float32)
# safeguard negatives or zeros - price should be non-negative
y_raw = np.where(np.isfinite(y_raw) & (y_raw >= 0), y_raw, 0.0)
y_log = np.log1p(y_raw)   # model will predict log1p(price)

# -------------------------
# 5) KFold with chunked training inside each fold
# -------------------------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
oof_preds_log = np.zeros_like(y_log, dtype=np.float32)
test_preds_log = np.zeros(test_img_features.shape[0], dtype=np.float32)

fold_smape_list = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(np.arange(len(y_log)))):
    print(f"\n===== Fold {fold+1}/{N_SPLITS} =====")
    # training indices for this fold
    tr_idx = np.array(tr_idx)
    val_idx = np.array(val_idx)

    # list of chunk-trained models for this fold
    fold_models = []

    # chunk over the training indices
    for start in range(0, len(tr_idx), CHUNK_SIZE):
        # indices in this chunk (relative to whole dataset)
        chunk_idx = tr_idx[start:start + CHUNK_SIZE]
        if len(chunk_idx) == 0:
            continue

        # build X_chunk on the fly (concatenate arrays for these indices)
        X_text_chunk = train_text_embs[chunk_idx]                     # (m, 768)
        X_img_chunk  = train_img_features[chunk_idx]                  # (m, 2048)
        X_text_feats_chunk = train_text_feats[chunk_idx]             # (m, 4)
        X_chunk = np.hstack([X_text_chunk, X_img_chunk, X_text_feats_chunk])  # (m, 2816+4)

        y_chunk = y_log[chunk_idx]

        # Train LightGBM regressor (no early stopping to keep compatibility)
        model = lgb.LGBMRegressor(**LGB_PARAMS)
        model.fit(X_chunk, y_chunk)   # small chunks ~15k rows each
        fold_models.append(model)

    # Predict OOF for val_idx by averaging models of this fold
    # Build X_val once (safe since val set is smaller than train)
    X_val_text = train_text_embs[val_idx]
    X_val_img  = train_img_features[val_idx]
    X_val_text_feats = train_text_feats[val_idx]
    X_val = np.hstack([X_val_text, X_val_img, X_val_text_feats])

    # average predictions in log-space, then store
    val_pred_logs = np.mean([m.predict(X_val) for m in fold_models], axis=0)
    oof_preds_log[val_idx] = val_pred_logs

    # compute SMAPE on original scale (inverse transform)
    val_pred = np.expm1(val_pred_logs)
    y_val_orig = np.expm1(y_log[val_idx])
    fold_sm = smape(y_val_orig, val_pred)
    fold_smape_list.append(fold_sm)
    print(f"Fold {fold+1} SMAPE: {fold_sm:.4f}%")

    # accumulate test predictions for this fold: average models predicting on test in log space
    # build test features in chunks to avoid mem explosion
    # but test_features size is manageable slice-by-slice; combine on the fly:
    X_test_combined = np.hstack([test_text_embs, test_img_features, test_text_feats])  # (n_test, D)
    fold_test_pred_logs = np.mean([m.predict(X_test_combined) for m in fold_models], axis=0)
    test_preds_log += fold_test_pred_logs / N_SPLITS

# -------------------------
# 6) Final OOF SMAPE and per-fold
# -------------------------
oof_preds = np.expm1(oof_preds_log)
overall_smape = smape(y_raw, oof_preds)
print("\nPer-fold SMAPE:", fold_smape_list)
print(f"Overall OOF SMAPE (on original price): {overall_smape:.4f}%")

# -------------------------
# 7) Build submission (inverse log1p)
# -------------------------
test_preds = np.expm1(test_preds_log)
# ensure positivity
test_preds = np.clip(test_preds, 0.0, None)

submission = pd.DataFrame({
    "sample_id": test_merged['sample_id'].values,   # from earlier merged frame
    "price": test_preds
})
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")


Loading files...
shapes:
 train_text_embs: (75000, 768)
 train_img_features: (75000, 2048)
 test_text_embs: (75000, 768)
 test_img_features: (75000, 2048)
Computing small text features...

===== Fold 1/5 =====


C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 1 SMAPE: 58.0690%


C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



===== Fold 2/5 =====


...


===== Fold 5/5 =====


C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 5 SMAPE: 57.3420%


C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



Per-fold SMAPE: [np.float64(58.06902426099359), np.float64(57.3047819717832), np.float64(57.041074601579965), np.float64(56.37871973808273), np.float64(57.34204820779373)]
Overall OOF SMAPE (on original price): 57.2271%
Submission saved to submission.csv


In [4]:
# ===============================
# 📌 STEP 0: Imports
# ===============================
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

# ===============================
# 📌 STEP 1: Load embeddings
# ===============================
train_text_embs = np.load("train_text_embeddings_fp16_v2.npy").astype(np.float32)
test_text_embs  = np.load("test_text_embeddings_fp16_v2.npy").astype(np.float32)

train_img_df = pd.read_csv("train_features.csv")
test_img_df  = pd.read_csv("test_features.csv")

# Ensure image embeddings are float32
img_cols = [c for c in train_img_df.columns if c != 'sample_id']
train_img_features = train_img_df[img_cols].astype(np.float32).values
test_img_features  = test_img_df[img_cols].astype(np.float32).values

# ===============================
# 📌 STEP 2: Align train/test sizes
# ===============================
min_train = min(train_text_embs.shape[0], train_img_features.shape[0])
train_text_embs = train_text_embs[:min_train]
train_img_features = train_img_features[:min_train]

min_test = min(test_text_embs.shape[0], test_img_features.shape[0])
test_text_embs = test_text_embs[:min_test]
test_img_features = test_img_features[:min_test]

# ===============================
# 📌 STEP 3: Combine embeddings
# ===============================
train_features = np.hstack([train_text_embs, train_img_features])
test_features  = np.hstack([test_text_embs, test_img_features])

# Optional: normalize features
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
test_features  = scaler.transform(test_features)

# Target
train_csv = pd.read_csv("dataset/train.csv")
y = train_csv['price'].values[:min_train]

# ===============================
# 📌 STEP 4: K-Fold + Chunked GPU Training
# ===============================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
chunk_size = 15000
oof_preds = np.zeros(len(train_features))
test_preds = np.zeros(len(test_features))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train_features)):
    print(f"\n🌟 Fold {fold+1}")
    
    X_tr, X_val = train_features[tr_idx], train_features[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    
    # Train in chunks
    models = []
    for i in range(0, len(X_tr), chunk_size):
        print(f"  Training chunk {i//chunk_size + 1}")
        X_chunk = X_tr[i:i+chunk_size]
        y_chunk = y_tr[i:i+chunk_size]
        
        model = lgb.LGBMRegressor(
            objective="regression",
            n_estimators=3000,
            learning_rate=0.01,
            num_leaves=256,
            feature_fraction=0.8,
            bagging_fraction=0.8,
            bagging_freq=5,
            min_data_in_leaf=50,
            lambda_l1=0.2,
            lambda_l2=0.4,
            device='gpu',      # <-- GPU enabled
            random_state=42
        )
        
        model.fit(
            X_chunk, y_chunk,
            eval_set=[(X_val, y_val)],
            eval_metric='mae',
            callbacks=[lgb.early_stopping(200)],
            verbose=100
        )
        models.append(model)
    
    # OOF prediction for this fold
    fold_oof = np.mean([m.predict(X_val) for m in models], axis=0)
    oof_preds[val_idx] = fold_oof
    
    # Test prediction for this fold
    fold_test = np.mean([m.predict(test_features) for m in models], axis=0)
    test_preds += fold_test / kf.n_splits

# ===============================
# 📌 STEP 5: Evaluate OOF SMAPE
# ===============================
def smape(y_true, y_pred):
    return 100/len(y_true) * np.sum(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

print("\n✅ OOF SMAPE:", smape(y, oof_preds))

# ===============================
# 📌 STEP 6: Clip predictions and Save submission
# ===============================
# Clip predictions to realistic target range
test_preds = np.clip(test_preds, 0, 50)  # as you suggested target <=50

submission = pd.DataFrame({
    'sample_id': test_img_df['sample_id'][:min_test],
    'price': test_preds
})

# Save with new name to not overwrite previous submission
submission.to_csv("submission_v2.csv", index=False)
print("✅ Submission saved as submission_v2.csv")




🌟 Fold 1
  Training chunk 1


TypeError: LGBMRegressor.fit() got an unexpected keyword argument 'verbose'

In [1]:
# ===============================
# 📌 STEP 0: Imports
# ===============================
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
import lightgbm as lgb
import os

# ===============================
# 📌 STEP 1: Load embeddings
# ===============================/
train_text_embs = np.load("train_text_embeddings_fp16_v2.npy").astype(np.float32)
test_text_embs  = np.load("test_text_embeddings_fp16_v2.npy").astype(np.float32)

train_img_df = pd.read_csv("train_features.csv")
test_img_df  = pd.read_csv("test_features.csv")

# Ensure image embeddings are float32
img_cols = [c for c in train_img_df.columns if c != 'sample_id']
train_img_features = train_img_df[img_cols].astype(np.float32).values
test_img_features  = test_img_df[img_cols].astype(np.float32).values

# ===============================
# 📌 STEP 2: Align train/test sizes
# ===============================
min_train = min(train_text_embs.shape[0], train_img_features.shape[0])
train_text_embs = train_text_embs[:min_train]
train_img_features = train_img_features[:min_train]

min_test = min(test_text_embs.shape[0], test_img_features.shape[0])
test_text_embs = test_text_embs[:min_test]
test_img_features = test_img_features[:min_test]

# ===============================
# 📌 STEP 3: Combine embeddings
# ===============================
train_features = np.hstack([train_text_embs, train_img_features])
test_features  = np.hstack([test_text_embs, test_img_features])

# Target column
train_csv = pd.read_csv("dataset/train.csv")
y = train_csv['price'].values[:min_train]

# ===============================
# 📌 STEP 4: K-Fold + Chunked GPU Training
# ===============================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
chunk_size = 15000
oof_preds = np.zeros(len(train_features))
test_preds = np.zeros(len(test_features))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train_features)):
    print(f"\n🌟 Fold {fold+1}")
    
    X_tr, X_val = train_features[tr_idx], train_features[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    
    models = []
    for i in range(0, len(X_tr), chunk_size):
        print(f"  Training chunk {i//chunk_size + 1}")
        X_chunk = X_tr[i:i+chunk_size]
        y_chunk = y_tr[i:i+chunk_size]
        
        model = lgb.LGBMRegressor(
            objective="regression",
            n_estimators=3000,
            learning_rate=0.05,
            num_leaves=64,
            device='gpu',       # ✅ Use GPU
            random_state=42
        )
        
        # Fit with GPU and early stopping
        model.fit(
            X_chunk, y_chunk,
            eval_set=[(X_val, y_val)],
            eval_metric='mae',
            callbacks=[
                lgb.early_stopping(stopping_rounds=200),
                lgb.log_evaluation(period=100)  # Logs every 100 rounds
            ]
        )
        models.append(model)
    
    # OOF prediction for this fold
    fold_oof = np.mean([m.predict(X_val) for m in models], axis=0)
    oof_preds[val_idx] = fold_oof
    
    # Test prediction for this fold
    fold_test = np.mean([m.predict(test_features) for m in models], axis=0)
    test_preds += fold_test / kf.n_splits

# ===============================
# 📌 STEP 5: Evaluate OOF SMAPE
# ===============================
def smape(y_true, y_pred):
    return 100/len(y_true) * np.sum(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

print("\n✅ OOF SMAPE:", smape(y, oof_preds))

# ===============================
# 📌 STEP 6: Save submission with new filename
# ===============================
submission_file = "submission_v2.csv"
submission = pd.DataFrame({
    'sample_id': test_img_df['sample_id'][:min_test],
    'price': test_preds
})
submission.to_csv(submission_file, index=False)
print(f"✅ Submission saved as {submission_file}")



🌟 Fold 1
  Training chunk 1
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 717545
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 2816
[LightGBM] [Info] Using GPU Device: Intel(R) UHD Graphics 770, Vendor: Intel(R) Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 1937 dense feature groups (27.75 MB) transferred to GPU in 0.017030 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 23.307043
Training until validation scores don't improve for 200 rounds
[100]	valid_0's l1: 17.2475	valid_0's l2: 974.412
[200]	valid_0's l1: 17.0863	valid_0's l2: 960.901
[300]	valid_0's l1: 17.0251	valid_0's l2: 956.523
[400]	valid_0's l1: 16.9906	valid_0's l2: 954.182
[500]	valid_0's l1: 16.97	valid_0's l2: 952.819
[600]	valid_0's l1: 16.9586	valid_0's l2: 952.112
[700]	valid_0's l1:

C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  


🌟 Fold 2
  Training chunk 1
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 717564
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 2816
[LightGBM] [Info] Using GPU Device: Intel(R) UHD Graphics 770, Vendor: Intel(R) Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 1940 dense feature groups (27.75 MB) transferred to GPU in 0.036088 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 23.510272
Training until validation scores don't improve for 200 rounds
[100]	valid_0's l1: 17.1763	valid_0's l2: 1049.93
[200]	valid_0's l1: 17.0053	valid_0's l2: 1035.57
[300]	valid_0's l1: 16.9227	valid_0's l2: 1028.07
[400]	valid_0's l1: 16.8857	valid_0's l2: 1024.99
[500]	valid_0's l1: 16.862	valid_0's l2: 1023.53
[600]	valid_0's l1: 16.8526	valid_0's l2: 1022.66
[700]	valid_0's l1

C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  


🌟 Fold 3
  Training chunk 1
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 717530
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 2816
[LightGBM] [Info] Using GPU Device: Intel(R) UHD Graphics 770, Vendor: Intel(R) Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 1937 dense feature groups (27.75 MB) transferred to GPU in 0.041311 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 23.464147
Training until validation scores don't improve for 200 rounds
[100]	valid_0's l1: 16.8091	valid_0's l2: 1300.9
[200]	valid_0's l1: 16.7021	valid_0's l2: 1289.39
[300]	valid_0's l1: 16.6367	valid_0's l2: 1284.49
[400]	valid_0's l1: 16.5938	valid_0's l2: 1281
[500]	valid_0's l1: 16.5759	valid_0's l2: 1279.34
[600]	valid_0's l1: 16.567	valid_0's l2: 1278.46
[700]	valid_0's l1: 16

...


🌟 Fold 4
  Training chunk 1
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 717544
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 2816
[LightGBM] [Info] Using GPU Device: Intel(R) UHD Graphics 770, Vendor: Intel(R) Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 1935 dense feature groups (27.69 MB) transferred to GPU in 0.032516 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 23.615819
Training until validation scores don't improve for 200 rounds
[100]	valid_0's l1: 16.8255	valid_0's l2: 826.748
[200]	valid_0's l1: 16.7061	valid_0's l2: 815.86
[300]	valid_0's l1: 16.6648	valid_0's l2: 811.622
[400]	valid_0's l1: 16.644	valid_0's l2: 809.211
[500]	valid_0's l1: 16.6357	valid_0's l2: 808.355
[600]	valid_0's l1: 16.6269	valid_0's l2: 807.655
[700]	valid_0's l1:

C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  


🌟 Fold 5
  Training chunk 1
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 717536
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 2816
[LightGBM] [Info] Using GPU Device: Intel(R) UHD Graphics 770, Vendor: Intel(R) Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 1938 dense feature groups (27.75 MB) transferred to GPU in 0.030557 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 23.334677
Training until validation scores don't improve for 200 rounds
[100]	valid_0's l1: 16.9587	valid_0's l2: 925.894
[200]	valid_0's l1: 16.7889	valid_0's l2: 910.449
[300]	valid_0's l1: 16.7299	valid_0's l2: 905.663
[400]	valid_0's l1: 16.6964	valid_0's l2: 903.425
[500]	valid_0's l1: 16.672	valid_0's l2: 902.039
[600]	valid_0's l1: 16.6661	valid_0's l2: 901.618
[700]	valid_0's l1

C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  


✅ OOF SMAPE: 69.81725413443547
✅ Submission saved as submission_v2.csv


In [ ]:
# ===============================
# 📌 STEP 0: Imports
# ===============================
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor, Pool
import os

# ===============================
# 📌 STEP 1: Load embeddings
# ===============================
train_text_embs = np.load("train_text_embeddings_fp16_v2.npy").astype(np.float32)
test_text_embs  = np.load("test_text_embeddings_fp16_v2.npy").astype(np.float32)

train_img_df = pd.read_csv("train_features.csv")
test_img_df  = pd.read_csv("test_features.csv")

# Ensure image embeddings are float32
img_cols = [c for c in train_img_df.columns if c != 'sample_id']
train_img_features = train_img_df[img_cols].astype(np.float32).values
test_img_features  = test_img_df[img_cols].astype(np.float32).values

# ===============================
# 📌 STEP 2: Align train/test sizes
# ===============================
min_train = min(train_text_embs.shape[0], train_img_features.shape[0])
train_text_embs = train_text_embs[:min_train]
train_img_features = train_img_features[:min_train]

min_test = min(test_text_embs.shape[0], test_img_features.shape[0])
test_text_embs = test_text_embs[:min_test]
test_img_features = test_img_features[:min_test]

# ===============================
# 📌 STEP 3: Combine embeddings
# ===============================
train_features = np.hstack([train_text_embs, train_img_features])
test_features  = np.hstack([test_text_embs, test_img_features])

# Target column
train_csv = pd.read_csv("dataset/train.csv")
y = train_csv['price'].values[:min_train]

# ===============================
# 📌 STEP 4: K-Fold + Chunked GPU Training
# ===============================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
chunk_size = 15000
oof_preds = np.zeros(len(train_features))
test_preds = np.zeros(len(test_features))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train_features)):
    print(f"\n🌟 Fold {fold+1}")
    
    X_tr, X_val = train_features[tr_idx], train_features[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    
    models = []
    for i in range(0, len(X_tr), chunk_size):
        print(f"  Training chunk {i//chunk_size + 1}")
        X_chunk = X_tr[i:i+chunk_size]
        y_chunk = y_tr[i:i+chunk_size]
        
        # ✅ GPU-optimized CatBoost
        model = CatBoostRegressor(
            iterations=3000,
            learning_rate=0.05,
            depth=8,
            loss_function="RMSE",   # GPU-compatible loss
            eval_metric="RMSE",
            task_type="GPU",
            devices='0',
            random_seed=42,
            verbose=100
        )
        
        # Train the model
        train_pool = Pool(X_chunk, y_chunk)
        val_pool = Pool(X_val, y_val)
        model.fit(train_pool, eval_set=val_pool)
        models.append(model)
    
    # OOF prediction for this fold
    fold_oof = np.mean([m.predict(X_val) for m in models], axis=0)
    oof_preds[val_idx] = fold_oof
    
    # Test prediction for this fold
    fold_test = np.mean([m.predict(test_features) for m in models], axis=0)
    test_preds += fold_test / kf.n_splits

# ===============================
# 📌 STEP 5: Evaluate OOF SMAPE
# ===============================
def smape(y_true, y_pred):
    return 100 / len(y_true) * np.sum(
        2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred))
    )

print("\n✅ OOF SMAPE:", smape(y, oof_preds))

# ===============================
# 📌 STEP 6: Save submission
# ===============================
submission_file = "submission_catboost_v3.csv"
submission = pd.DataFrame({
    'sample_id': test_img_df['sample_id'][:min_test],
    'price': test_preds
})
submission.to_csv(submission_file, index=False)
print(f"✅ Submission saved as {submission_file}")



🌟 Fold 1
  Training chunk 1


In [3]:
!pip install catboost

   ---------------------------------------- 0.0/102.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/102.5 MB ? eta -:--:--
   ---------------------------------------- 0.8/102.5 MB 2.2 MB/s eta 0:00:46
    --------------------------------------- 1.6/102.5 MB 2.6 MB/s eta 0:00:39
    --------------------------------------- 2.4/102.5 MB 3.0 MB/s eta 0:00:33
   - -------------------------------------- 3.4/102.5 MB 3.5 MB/s eta 0:00:29
   - -------------------------------------- 4.2/102.5 MB 3.6 MB/s eta 0:00:28
   - -------------------------------------- 5.0/102.5 MB 3.6 MB/s eta 0:00:28
   -- ------------------------------------- 5.8/102.5 MB 3.5 MB/s eta 0:00:28
   -- ------------------------------------- 6.6/102.5 MB 3.6 MB/s eta 0:00:27
   -- ------------------------------------- 7.6/102.5 MB 3.6 MB/s eta 0:00:27
   --- ------------------------------------ 8.4/102.5 MB 3.7 MB/s eta 0:00:26
   --- ------------------------------------ 9.4/102.5 MB 3.8 MB/s eta 0:00:25


In [1]:
from catboost import CatBoostRegressor, Pool


In [3]:
!pip install catboost


In [4]:
!pip install catboost optuna


   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ------------------- -------------------- 1.0/2.1 MB 7.2 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 5.2 MB/s  0:00:00

   ---------------------------------------- 0/6 [Mako]
   ------ --------------------------------- 1/6 [greenlet]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalche

In [ ]:
# ===============================
# 📌 STEP 0: Imports
# ===============================
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor, Pool
import optuna
import os

# ===============================
# 📌 STEP 1: Load embeddings
# ===============================
train_text_embs = np.load("train_text_embeddings_fp16_v2.npy").astype(np.float32)
test_text_embs  = np.load("test_text_embeddings_fp16_v2.npy").astype(np.float32)

train_img_df = pd.read_csv("train_features.csv")
test_img_df  = pd.read_csv("test_features.csv")

# Ensure image embeddings are float32
img_cols = [c for c in train_img_df.columns if c != 'sample_id']
train_img_features = train_img_df[img_cols].astype(np.float32).values
test_img_features  = test_img_df[img_cols].astype(np.float32).values

# ===============================
# 📌 STEP 2: Align train/test sizes
# ===============================
min_train = min(train_text_embs.shape[0], train_img_features.shape[0])
train_text_embs = train_text_embs[:min_train]
train_img_features = train_img_features[:min_train]

min_test = min(test_text_embs.shape[0], test_img_features.shape[0])
test_text_embs = test_text_embs[:min_test]
test_img_features = test_img_features[:min_test]

# ===============================
# 📌 STEP 3: Combine embeddings
# ===============================
train_features = np.hstack([train_text_embs, train_img_features])
test_features  = np.hstack([test_text_embs, test_img_features])

# Target column
train_csv = pd.read_csv("dataset/train.csv")
y = train_csv['price'].values[:min_train]

# ===============================
# 📌 STEP 4: Define SMAPE metric
# ===============================
def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0.0
    return np.mean(diff) * 100

# ===============================
# 📌 STEP 5: Optuna objective function
# ===============================
def objective(trial):
    params = {
        "iterations": 1500,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 5),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "random_strength": trial.suggest_float("random_strength", 1, 20),
        "loss_function": "RMSE",
        "task_type": "GPU",
        "devices": "0",
        "verbose": 0,
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    smape_scores = []
    for train_idx, val_idx in kf.split(train_features):
        X_train, X_val = train_features[train_idx], train_features[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        train_pool = Pool(X_train, y_train)
        val_pool   = Pool(X_val, y_val)

        model = CatBoostRegressor(**params)
        model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=100, verbose=False)
        preds = model.predict(X_val)
        smape_scores.append(smape(y_val, preds))
    
    return np.mean(smape_scores)

# ===============================
# 📌 STEP 6: Run Optuna tuning
# ===============================
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15, show_progress_bar=True)  # Try 15–20 for faster tuning

print("\n🎯 Best SMAPE:", study.best_value)
print("📊 Best Params:", study.best_params)

# ===============================
# 📌 STEP 7: Retrain best model on all data
# ===============================
best_params = study.best_params
best_params.update({
    "iterations": 2000,
    "task_type": "GPU",
    "devices": "0",
    "loss_function": "RMSE",
    "verbose": 200
})

final_model = CatBoostRegressor(**best_params)
final_model.fit(train_features, y)

# ===============================
# 📌 STEP 8: Predict and save submission
# ===============================
test_preds = final_model.predict(test_features)
submission = pd.DataFrame({
    "sample_id": test_img_df["sample_id"][:min_test],
    "price": test_preds
})
submission.to_csv("submission_catboost_tuned.csv", index=False)

print("\n✅ Tuning complete! Best SMAPE:", study.best_value)
print("✅ Submission saved as submission_catboost_tuned.csv")


[I 2025-10-12 17:52:45,454] A new study created in memory with name: no-name-14fbf8bb-7a2c-42f3-9973-5423134e26a2


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2025-10-12 18:02:17,016] Trial 0 finished with value: 69.19797346910151 and parameters: {'learning_rate': 0.0788112840599579, 'depth': 5, 'l2_leaf_reg': 3.331871793104847, 'bagging_temperature': 1.4451343576979019, 'border_count': 141, 'random_strength': 17.697668092614606}. Best is trial 0 with value: 69.19797346910151.
